This notebook generates base `meta-llama/Llama-3.1-8B-Instruct` outputs from `data/full_silver-standard_dataset.csv`, converting each silver-standard panel prompt into a Markdown blood-test table before generation.


### Install Dependencies


In [ ]:
# Optional: uncomment if needed
# %pip install -r ../../requirements.txt

### Configure Paths And Token



In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "llama":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

INPUT_CSV = PROJECT_ROOT / "data" / "full_silver-standard_dataset.csv"
OUTPUT_CSV = PROJECT_ROOT / "llama" / "outputs" / "tabular_prompt_approach" / "llama_full_silver_base_outputs.csv"
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MAX_ROWS = 10
MAX_NEW_TOKENS = 900
USE_4BIT = True

# Set your token here for this notebook session, or set HF_TOKEN in your shell before launching Jupyter.
# os.environ["HF_TOKEN"] = "hf_your_token_here"

print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_CSV)
print("Output:", OUTPUT_CSV)
print("HF_TOKEN set:", bool(os.getenv("HF_TOKEN")))

Project root: c:\Users\dimit\OneDrive\desktop\NLP_medical_results_explanation
Input: c:\Users\dimit\OneDrive\desktop\NLP_medical_results_explanation\data\mimic_labs_20_for_testing.csv
Output: c:\Users\dimit\OneDrive\desktop\NLP_medical_results_explanation\llama\outputs\llama_tabular_base_outputs.csv
HF_TOKEN set: False


### Load The Same Raw Input Rows As For Text-only Prompt



In [2]:
import pandas as pd

df = pd.read_csv(INPUT_CSV)
df = df.head(MAX_ROWS).copy()
print("Rows loaded:", len(df))
df.head(3)

Rows loaded: 10


,SUBJECT_ID,HADM_ID,ITEMID,lab_name,fluid,category,loinc_code,CHARTTIME,VALUE,VALUENUM,VALUEUOM,FLAG,GENDER,ADMISSION_TYPE,DIAGNOSIS,input_text,target_text
0,585,123154,50820,pH,Blood,Blood Gas,11558-4,2142-11-10 17:41:00,7.32,7.32,units,abnormal,F,EMERGENCY,STATUS POST MOTOR VEHICLE ACCIDENT WITH INJURIES,Patient sex: F\r\nAdmission type: EMERGENCY\r\...,NaN
1,585,123154,50821,pO2,Blood,Blood Gas,11556-8,2142-11-10 17:41:00,236.00,236.00,mm Hg,abnormal,F,EMERGENCY,STATUS POST MOTOR VEHICLE ACCIDENT WITH INJURIES,Patient sex: F\r\nAdmission type: EMERGENCY\r\...,NaN
2,585,123154,50867,Amylase,Blood,Chemistry,1798-8,2142-11-10 19:58:00,64.00,64.00,IU/L,NaN,F,EMERGENCY,STATUS POST MOTOR VEHICLE ACCIDENT WITH INJURIES,Patient sex: F\r\nAdmission type: EMERGENCY\r\...,NaN


### Build Table-Style Prompts



In [4]:
from llama.scripts.tabular_approach.prompt_utils import SYSTEM_PROMPT, build_tabular_prompt

print("Using prompt builder:", build_tabular_prompt.__name__)

In [5]:
example_prompt = build_tabular_prompt(df.iloc[0])
print(example_prompt)

Read the laboratory result in the table below.

| field | value |
| --- | --- |
| Patient sex | F |
| Admission type | EMERGENCY |
| Admission diagnosis | STATUS POST MOTOR VEHICLE ACCIDENT WITH INJURIES |
| Laboratory test | pH |
| Fluid | Blood |
| Category | Blood Gas |
| Measured value | 7.32 units |
| Abnormal flag | abnormal |

Task: Write one short, patient-friendly explanation of what this result may mean for the body.
Rules:
- Use simple language.
- Use cautious wording such as may suggest, can suggest, may reflect, or appears.
- Do not diagnose disease.
- Do not recommend treatment.
- Keep the answer to one or two short sentences.


### Load Base Llama

This is the first model-heavy cell. It requires a Hugging Face token with access to `meta-llama/Llama-3.1-8B-Instruct`. With `USE_4BIT=True`, it also requires CUDA and `bitsandbytes`.

In [10]:
import torch
import transformers.utils.import_utils as transformers_import_utils
import transformers.utils as transformers_utils
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

transformers_import_utils.is_torchvision_available = lambda: False
transformers_utils.is_torchvision_available = lambda: False

token = (os.getenv("HF_TOKEN") or "").strip()
if not token:
    raise EnvironmentError("Set HF_TOKEN before loading Llama weights from Hugging Face.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

supports_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
quantization_config = None

if USE_4BIT:
    if not torch.cuda.is_available():
        raise RuntimeError("4-bit generation requires a CUDA GPU. Set USE_4BIT = False for CPU/full precision.")
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=token,
    dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
    quantization_config=quantization_config,
    device_map={"": 0} if USE_4BIT else "auto",
)
model.eval()
print("Loaded", MODEL_ID)

c:\Users\dimit\OneDrive\Desktop\NLP_medical_results_explanation\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dimit\.cache\huggingface\hub\models--meta-llama--Llama-3.1-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


RuntimeError: 4-bit generation requires a CUDA GPU. Set USE_4BIT = False for CPU/full precision.

### Generate One Example

In [ ]:
def build_chat_prompt(user_prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate(prompt, max_new_tokens=MAX_NEW_TOKENS):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

chat_prompt = build_chat_prompt(example_prompt)
one_output = generate(chat_prompt)
print(one_output)

### Generate And Save All Rows

In [ ]:
df["base_llama_tabular_prompt"] = ""
df["base_llama_tabular_output"] = ""

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

for row_index in range(len(df)):
    tabular_prompt = build_tabular_prompt(df.iloc[row_index])
    model_prompt = build_chat_prompt(tabular_prompt)
    df.at[row_index, "base_llama_tabular_prompt"] = model_prompt
    df.at[row_index, "base_llama_tabular_output"] = generate(model_prompt)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"Processed row {row_index + 1}/{len(df)}")

print("Wrote:", OUTPUT_CSV)
preview_columns = [
    column
    for column in ["summary_id", "subject_id", "model_used", "generated_text", "base_llama_tabular_output"]
    if column in df.columns
]
df[preview_columns].head()